# 2 Autoencoders

[**Autoencoder (2006):**](https://www.cs.toronto.edu/~hinton/absps/science.pdf) NN used for unsupervised learning. 

**Objective:** To learn a compressed, efficient representation of input data without external labels.

Instead of predicting a target label $y$ from an input $x$, an autoencoder is trained to reconstruct its own input ($x \to x'$) after passing it through a constrained internal bottleneck.

An autoencoder consists of three main components:

1. **Encoder ($f_\theta$):** NN that compresses high-dimensional input data $x$ into a lower-dimensional latent representation $z$.

$$z = f_\theta(x)$$


2. **Bottleneck (Latent Space, $z$):** Low-dimensional layer in the network to force to ignore noise and preserve most important features of data.

3. **Decoder ($g_\phi$):** NN that takes the latent *code* $z$ and attempts to reconstruct the original input as $\hat{x}$.

$$\hat{x} = g_\phi(z)$$




<p style="page-break-after:always;"></p>

### Objective functions in training

NN learns to minimize a *reconstruction* loss ($\mathcal{L}$). 

The reconstruction loss measures the difference between the input $x$ and the reconstructed output $\hat{x}$. 

**Mean Squared Error (MSE)** 

<!--
Commonly used for continuous real-valued inputs (like images normalized between 0 and 1).

For a single sample with $D$ feature dimensions (e.g., $D$ pixels in an image), the total squared reconstruction loss is:
-->

$$\mathcal{L}_{\text{MSE}}(x, \hat{x}) = \frac{1}{D} \sum_{i=1}^{D} (x_i - \hat{x}_i)^2$$

where

* $x_i$ is the original value of feature $i$.
* $\hat{x}_i$ is the reconstructed value of feature $i$ (produced by the decoder).
* $D$ is the total number of input dimensions/features.

**Binary Cross-Entropy (BCE)** 

<!--
Often used when inputs are treated as Bernoulli distributions or binary values.

In autoencoders, Binary Cross-Entropy (BCE) measures the difference between the true input values $x$ and the reconstructed output values $\hat{x}$. It is typically used when the input data features are normalized to a continuous range of $[0, 1]$ (or are binary) and treated as probabilities. For a single sample with $D$ feature dimensions (e.g., $D$ pixels in an image), the Binary Cross-Entropy loss is:
-->

$$\mathcal{L}_{\text{BCE}}(x, \hat{x}) = -\sum_{i=1}^{D} \left[ x_i \log(\hat{x}_i) + (1 - x_i) \log(1 - \hat{x}_i) \right]$$

where

* $x_i \in [0, 1]$ is the original value of feature $i$.
* $\hat{x}_i \in (0, 1)$ is the reconstructed value of feature $i$ (usually produced by a sigmoid output layer).
* $D$ is the total number of input dimensions/features.



<p style="page-break-after:always;"></p>

### Variants

**Undercomplete AE**

Latent dimension is smaller than input dimension for basic dimensionality reduction and compression.

**[Denoising AE (DAE)](https://www.cs.toronto.edu/~larocheh/publications/icml-2008-denoising-autoencoders.pdf)**

Adds noise to inputs during training to prevent learning an identity mapping

The decoder is required to reconstruct the original, uncorrupted data learning to strip noise away

*Objective*: Noise removal and robust feature extraction

*Corruption:* An original input $x$ is stochastically corrupted using a corruption distribution $q(\tilde{x} \vert{} x)$ to produce a noisy input $\tilde{x}$. Common corruption strategies

* Additive Gaussian Noise: Adds random noise: $\tilde{x} = x + \epsilon$, where $\epsilon \sim \mathcal{N}(0, \sigma^2 I)$

* Masking: Randomly sets a fraction $\nu$ of input elements to 0 (or min/max values)

* Dropout Noise: Randomly zeros out features with probability $p$ during the forward pass

<p style="page-break-after:always;"></p>

**[Sparse AE](https://web.stanford.edu/class/cs294a/sparseAutoencoder_2011new.pdf)**

Forces hidden units to be mostly inactive (close to zero) via regularization penalties for any given input

*Objective*: Feature extraction without shrinking layer size

<!--
Unlike undercomplete autoencoders, a sparse autoencoder can actually have an overcomplete latent dimension (larger than the input dimension)

The sparsity constraint ensures that even with a large capacity, the network learns meaningful, disentangled representations rather than trivial identity mappings
-->

*Steps:*

* Forward pass: Input $x$ is passed through the encoder to obtain hidden activation vector $h = f_\theta(x)$

* Activation tracking: Across a mini-batch of N samples, the average activation $\hat{\rho}_j$ of each hidden neuron $j$ is calculated as

$$\hat{\rho}_j = \frac{1}{N} \sum_{i=1}^{N} a_j(x^{(i)})$$

where $a_j(x^{(i)})$ is the activation value of hidden unit $j$ when processing input $x^{(i)}$ (i.e. sigmoid in $(0, 1)$).


* Sparsity Penalty: A penalty term is added to the loss function that penalizes any hidden unit whose average activation deviates from a small target threshold $\rho$ (e.g., $\rho = 0.05$, meaning neurons should be active only 5% of the time)

$$\mathcal{L}_{\text{SAE}}(\theta, \phi) = \mathcal{L}(x, \hat{x}) + \beta \sum_{j=1}^{K} \text{Penalty}(\rho, \hat{\rho}_j) + \lambda \Omega(W)$$

where

* $\mathcal{L}$ is MSE or BCE loss function.
* $K$ is the total number of hidden units in the latent layer.
* $\rho$ is the target sparsity parameter (a value close to $0$).
* $\hat{\rho}_j$ is the empirical average activation of neuron $j$.
* $\beta$ is a hyperparameter controlling the weight of the sparsity penalty.
* $\lambda \Omega(W)$ is an optional weight decay regularizer ($L_2$ norm) to prevent overfitting.

*Sparsity penalty functions*

* Kullback-Leibler (KL) divergence: difference between two Bernoulli distributions, one with mean $\rho$ (target) and one with mean $\hat{\rho}_j$ (actual):

$$\text{KL}(\rho \parallel \hat{\rho}_j) = \rho \log \left( \frac{\rho}{\hat{\rho}_j} \right) + (1 - \rho) \log \left( \frac{1 - \rho}{1 - \hat{\rho}_j} \right)$$

* $L_1$ regularization penalty can be directly applied to the latent activations $h$:

$$\text{L1}(h) = \sum_{j=1}^{K} \vert{}h_j\vert{}$$

*Key Benefits*

* Feature interpretability: Each hidden unit specializes in detecting a distinct, highly specific subfeature

* Overcomplete capacities: Allows the latent dimension to be larger than the input dimension ($K > D$) without overfitting or degenerating into an identity copy operation.

* Modern LLM interpretability: Modern mechanistic interpretability uses massive sparse autoencoders to extract human-understandable concepts from the hidden activations of Transformer models.

<p style="page-break-after:always;"></p>

**[Variational AE](https://arxiv.org/pdf/1312.6114)** 

Unlike conventional autoencoders that map input data $x$ to a single deterministic point in latent space, a VAE maps $x$ into a probability distribution over the latent space $z$. 

This allows VAE to generate entirely new samples by sampling random vectors $z$ from a prior distribution (such as a standard normal distribution $\mathcal{N}(0, I)$) and passing them through the decoder.

**Objective:** Maximize the marginal log-likelihood of observed data $\log p_\theta(x)$

$$
\hat{\theta} = \operatorname{argmax}_\theta \log p_\theta(x)
$$

Because $x$ depends on an unobserved (latent) variable $z$, the marginal probability is defined by integrating over all possible latent values $z$

$$
p_\theta(x) = \int p_\theta(x, z) \, dz = \int p_\theta(z \mid x) \, p_\theta(x) \, dz = \mathbb{E}_{p_\theta(z\vert{}x)} \left[ \log p_\theta(x) \right]
$$

Evaluating this integral directly is computationally intractable for complex neural networks. We introduce an **inference network (encoder)** $q_\phi(z\vert{}x)$ to approximate the true posterior $p_\theta(z\vert{}x)$

$$
\log p_\theta(x) = \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x) \right]
$$

Using the definition of conditional probability, we can rewrite the joint distribution $p_\theta(x, z)$ as

$$p_\theta(x) = \frac{p_\theta(x, z)}{p_\theta(z\vert{}x)}$$

Substituting this expression into the expectation

$$\log p_\theta(x) = \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x, z)}{p_\theta(z\vert{}x)} \right) \right]$$


Multiplying the numerator and denominator inside the logarithm by $q_\phi(z\vert{}x)$

$$\begin{aligned}
\log p_\theta(x) 
&= \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x, z)}{p_\theta(z\vert{}x)} \cdot \frac{q_\phi(z\vert{}x)}{q_\phi(z\vert{}x)} \right) \right]\\
&= \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x, z)}{q_\phi(z\vert{}x)} \cdot \frac{q_\phi(z\vert{}x)}{p_\theta(z\vert{}x)} \right) \right]\\
&= \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x, z)}{q_\phi(z\vert{}x)} \right) + \log \left( \frac{q_\phi(z\vert{}x)}{p_\theta(z\vert{}x)} \right) \right]\\
&= \underbrace{\mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x, z)}{q_\phi(z\vert{}x)} \right) \right]}_{\text{Term 1: ELBO}} + \underbrace{\mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{q_\phi(z\vert{}x)}{p_\theta(z\vert{}x)} \right) \right]}_{\text{Term 2: KL Divergence}}
\end{aligned}$$


Term 1 is defined as the **Evidence Lower Bound (ELBO)**, denoted as $\mathcal{L}_{\text{ELBO}}(\theta, \phi; x)$.

By definition, the Kullback-Leibler (KL) divergence between two continuous distributions $q(z)$ and $p(z)$ is

$$D_{\text{KL}}(q \parallel p) = \int q(z) \log \left( \frac{q(z)}{p(z)} \right) dz = \mathbb{E}_{q} \left[ \log \left( \frac{q(z)}{p(z)} \right) \right]$$

Therefore, Term 2 is precisely the KL divergence between our approximate posterior $q_\phi(z\vert{}x)$ and the true posterior $p_\theta(z\vert{}x)$

$$\mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{q_\phi(z\vert{}x)}{p_\theta(z\vert{}x)} \right) \right] = D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p_\theta(z\vert{}x) \right)$$

Putting both terms back together gives the fundamental identity

$$\log p_\theta(x) = \mathcal{L}_{\text{ELBO}}(\theta, \phi; x) + D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p_\theta(z\vert{}x) \right)$$


<p style="page-break-after:always;"></p>

The goal is to maximize the marginal log-likelihood of observed data $\log p_\theta(x)$

$$\begin{aligned}
\log p_\theta(x) &= \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x) \right] \\
 &= \underbrace{\mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \frac{p_\theta(x, z)}{q_\phi(z\vert{}x)} \right]}_{\text{ELBO}(\theta, \phi; x)} + \underbrace{D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p_\theta(z\vert{}x) \right)}_{\ge 0}
\end{aligned}$$

Since the Kullback-Leibler (KL) divergence is always non-negative ($D_{\text{KL}} \ge 0$), the ELBO serves as a lower bound on data log-likelihood

$$\log p_\theta(x) \ge \mathcal{L}_{\text{ELBO}}(\theta, \phi; x)$$

Alternatively, by maximizing $\mathcal{L}_{\text{ELBO}}(\theta, \phi; x)$

$$\begin{aligned}
\left(\hat{\theta},\, \hat{\phi}\right) 
&= \operatorname{argmax}_{\left (\theta,\, \phi\right)} \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \frac{p_\theta(x, z)}{q_\phi(z\vert{}x)} \right]\\
&= \operatorname{argmax}_{\left (\theta,\, \phi\right)}  \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x\vert{}z) \right] - D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p(z) \right)
\end{aligned}$$

## Summary of the Final Formulation

| Term | Mathematical Expression | Intuition |
| --- | --- | --- |
| **Reconstruction Loss** | $\mathbb{E}_{q_\phi(z\vert{}x)} [\log p_\theta(x\Vert{}z)]$ | Measures how accurately the decoder $p_\theta(x\Vert{}z)$ reconstructs $x$ from latent code $z$. |
| **KL Regularization** | $D_{\text{KL}}(q_\phi(z\Vert{}x) \parallel p(z))$ | Measures how much the encoded distribution $q_\phi(z\Vert{}x)$ deviates from the prior $p(z) = \mathcal{N}(0, I)$. |

Since neural networks minimize loss via gradient descent, we negate the ELBO to get our final training objective:

$$\text{Loss}_{\text{VAE}}(\theta, \phi; x) = -\mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x\vert{}z) \right] + D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p(z) \right)$$


To understand why evaluating $p_\theta(x) = \int p_\theta(x, z) \, dz$ is computationally intractable, we have to look at what that integral actually represents and why traditional numerical methods completely break down.

---

## 1. The High-Dimensionality Trap (Curse of Dimensionality)

The latent variable $z$ is rarely a 1D or 2D scalar; in modern deep learning models, $z$ is a high-dimensional vector (e.g., $d = 128$, $512$, or $2048$ dimensions).

To compute the integral $\int p_\theta(x, z) \, dz$, you would theoretically need to integrate over the **entire continuous space** of $z$:

$$p_\theta(x) = \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} \dots \int_{-\infty}^{\infty} p_\theta(x \vert{} z_1, z_2, \dots, z_d) \, p(z_1, z_2, \dots, z_d) \, dz_1 \, dz_2 \dots dz_d$$

### Numerical Quadrature Fails

If you tried standard numerical integration (like grid approximation) using just 10 evaluation points per dimension:

* For a 2D latent space: $10^2 = 100$ evaluations.
* For a 100D latent space: $10^{100}$ evaluations.

$10^{100}$ is greater than the total number of atoms in the observable universe. Evaluating a neural network that many times for a single data point $x$ is physically impossible.

---

## 2. Naïve Monte Carlo Sampling is Exhaustively Inefficient

You might think: *"What if we don't use a grid, but instead approximate the integral by sampling random values of $z$ from the prior $p(z) = \mathcal{N}(0, I)$?"*

Using Simple Monte Carlo estimation:

$$p_\theta(x) = \int p_\theta(x\vert{}z) p(z) \, dz \approx \frac{1}{S} \sum_{s=1}^{S} p_\theta(x \vert{} z^{(s)}), \quad \text{where } z^{(s)} \sim p(z)$$

While this solves the exponential grid explosion, **it introduces a needle-in-a-haystack problem**:

```
 High-Dimensional Latent Space p(z)
┌──────────────────────────────────────────────┐
│  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . │
│  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . │
│  .  .  .  .  .  [Tiny region] .  .  .  .  . │  <-- Only z values inside this tiny 
│  .  .  .  .  .  [where p(x|z)] .  .  .  .  . │      region produce a realistic image x.
│  .  .  .  .  .  [  is > 0   ] .  .  .  .  . │      Everything else outputs nonsense 
│  .  .  .  .  .  .  .  .  .  .  .  .  .  .  . │      (p(x|z) ≈ 0).
└──────────────────────────────────────────────┘

```

1. For a complex image $x$ (e.g., a specific face), **almost all** randomly sampled $z \sim \mathcal{N}(0, I)$ will decode into useless noise.
2. The conditional likelihood $p_\theta(x\vert{}z)$ will evaluate to virtually **$0$** for almost all $z$.
3. Only an infinitesimal, narrow volume of the latent space contains $z$-vectors that actually reconstruct $x$.
4. Randomly hitting that tiny region in high dimensions by chance requires an extraordinarily large number of samples $S$, leading to huge variance and failure to converge.

---

## 3. The Non-Linearity of Deep Neural Networks

If $p_\theta(x\vert{}z)$ were a simple linear transformation or a simple Gaussian conjugate, we could compute the integral analytically on paper using standard calculus rules.

However, in deep autoencoders, $p_\theta(x\vert{}z)$ is parameterized by a **deep neural network** with non-linear activation functions (ReLU, GELU, Sigmoid, attention layers, matrix multiplications):

$$p_\theta(x\vert{}z) = \mathcal{N}\Big(x; \, \text{Decoder}_{\theta}(z), \, \sigma^2 I\Big)$$

Because $\text{Decoder}_{\theta}(z)$ is a highly complex, non-linear composition of functions, the integrand $p_\theta(x\vert{}z)p(z)$ has no closed-form antiderivative. There is no mathematical shortcut to integrate through non-linear neural network layers analytically.

---

## The Solution: Amortized Variational Inference

Because direct evaluation is impossible, Variational Autoencoders reframe the problem:

> Instead of searching the entire latent space blindly for valid $z$'s, we train an **encoder network** $q_\phi(z\vert{}x)$ that directly learns to predict: *"Given this specific input $x$, which region of $z$-space is actually responsible for it?"*

By sampling $z$ strictly from $q_\phi(z\vert{}x)$ instead of the prior $p(z)$, we sample only from the "needle" region where $p_\theta(x\vert{}z) > 0$, turning an intractable integration problem into a fast, differentiable optimization problem (the ELBO).

A **Variational Autoencoder (VAE)** (Kingma & Welling, 2013) is a probabilistic generative model.

Unlike standard autoencoders that map input data $x$ to a single deterministic point in latent space, a VAE maps $x$ into a **probability distribution** over the latent space $z$. This allows VAEs to generate entirely new samples by sampling random vectors $z$ from a prior distribution (such as a standard normal distribution $\mathcal{N}(0, I)$) and passing them through the decoder.

---

## 1. Probabilistic Graphical Model

A VAE models the data generation process as follows:

1. **Prior:** A latent variable $z$ is drawn from a prior distribution $p(z) = \mathcal{N}(0, I)$.
2. **Generative Model (Decoder):** An observation $x$ is sampled from a conditional likelihood $p_\theta(x\vert{}z)$, parametrized by a neural network with parameters $\theta$.

Because the true posterior $p_\theta(z\vert{}x) = \frac{p_\theta(x\vert{}z)p(z)}{p_\theta(x)}$ is computationally intractable to calculate directly, a VAE uses an **inference model (Encoder)** $q_\phi(z\vert{}x)$ to approximate $p_\theta(z\vert{}x)$.

---

## 2. Mathematical Derivation & Objective (ELBO)

The goal is to maximize the marginal log-likelihood of observed data $\log p_\theta(x)$. By applying Bayes' rule and Jensen's inequality, we derive the **Evidence Lower Bound (ELBO)** $\mathcal{L}_{\text{ELBO}}(\theta, \phi; x)$:

$$\begin{aligned} \log p_\theta(x) &= \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x) \right] \\ &= \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \frac{p_\theta(x, z)}{q_\phi(z\vert{}x)} \cdot \frac{q_\phi(z\vert{}x)}{p_\theta(z\vert{}x)} \right] \\ &= \underbrace{\mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \frac{p_\theta(x, z)}{q_\phi(z\vert{}x)} \right]}_{\text{ELBO}(\theta, \phi; x)} + \underbrace{D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p_\theta(z\vert{}x) \right)}_{\ge 0} \end{aligned}$$

Since the Kullback-Leibler (KL) divergence is always non-negative ($D_{\text{KL}} \ge 0$), the ELBO serves as a lower bound on data log-likelihood:

$$\log p_\theta(x) \ge \mathcal{L}_{\text{ELBO}}(\theta, \phi; x)$$

Maximizing the ELBO both maximizes the likelihood of generating observed data and forces the approximate posterior $q_\phi(z\vert{}x)$ closer to the true posterior $p_\theta(z\vert{}x)$.

---

## 3. The Loss Function

In practice, neural networks are trained to **minimize the negative ELBO**. Expanding the ELBO gives the two-part loss function:

$$\mathcal{L}_{\text{VAE}}(\theta, \phi; x) = -\mathcal{L}_{\text{ELBO}} = \underbrace{-\mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x\vert{}z) \right]}_{\text{Reconstruction Loss}} + \underbrace{D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p(z) \right)}_{\text{KL Regularization Loss}}$$

### Component Breakdown

1. **Reconstruction Loss:** Measures how well the decoder reconstructs input $x$ given latent variable $z$.
* Computed via **Mean Squared Error (MSE)** for continuous variables or **Binary Cross-Entropy (BCE)** for normalized/binary variables.


2. **KL Regularization Loss:** Penalizes divergence between the approximate posterior $q_\phi(z\vert{}x)$ and the standard prior distribution $p(z) = \mathcal{N}(0, I)$. This forces the latent space to stay smooth, continuous, and centered at the origin.

---

### Closed-Form Solution for Gaussian Latent Variables

Assuming $p(z) = \mathcal{N}(0, I)$ and $q_\phi(z\vert{}x) = \mathcal{N}(\mu_x, \text{diag}(\sigma_x^2))$, the KL divergence term can be solved analytically:

$$D_{\text{KL}}\left( \mathcal{N}(\mu_x, \sigma_x^2) \parallel \mathcal{N}(0, I) \right) = -\frac{1}{2} \sum_{j=1}^{J} \left( 1 + \log(\sigma_j^2) - \mu_j^2 - \sigma_j^2 \right)$$

Where $J$ is the number of latent dimensions, and $\mu_j, \sigma_j$ are the predicted mean and standard deviation for latent dimension $j$.

---

## 4. The Reparameterization Trick

To train the network end-to-end using backpropagation, gradients must flow through the latent layer $z \sim q_\phi(z\vert{}x)$. Sampling directly introduces a non-differentiable stochastic node.

To fix this, Kingma & Welling introduced the **reparameterization trick**, expressing sampling deterministically by isolating the randomness into an auxiliary noise variable $\epsilon$:

$$\epsilon \sim \mathcal{N}(0, I)$$

$$z = \mu_\phi(x) + \sigma_\phi(x) \odot \epsilon$$

Where $\odot$ denotes element-wise multiplication.

```
       [ Input x ]
            │
      ┌─────┴─────┐
      │  Encoder  │
      └─────┬─────┘
            │
   ┌────────┴────────┐
   ▼                 ▼
 Mean (μ)       Log Variance (log σ²)
   │                 │
   │    ┌────────────┘  ┌────────────────┐
   │    │               │  Noise ε ~ N(0, I)
   ▼    ▼               └───────┬────────┘
  [ z = μ + σ ⊙ ε ] ◄───────────┘  (Differentiable Sampling)
        │
  ┌─────┴─────┐
  │  Decoder  │
  └─────┬─────┘
        ▼
   [ Reconstructed x̂ ]

```

Because stochasticity is isolated inside $\epsilon$, gradients can flow back smoothly through $\mu_\phi(x)$ and $\sigma_\phi(x)$ during backpropagation.

---

Here is the full step-by-step mathematical derivation of the **Evidence Lower Bound (ELBO)**, detailing every algebraic step and property used.

---

# Detailed Derivation of the ELBO

Our goal is to compute or maximize the marginal log-likelihood of our observed data $x$:

$$\log p_\theta(x)$$

Because $x$ depends on an unobserved (latent) variable $z$, the marginal probability is defined by integrating over all possible latent states $z$:

$$p_\theta(x) = \int p_\theta(x, z) \, dz$$

Evaluating this integral directly is computationally intractable for complex neural networks. We introduce an **inference network (encoder)** $q_\phi(z\vert{}x)$ to approximate the true posterior $p_\theta(z\vert{}x)$.

---

## Step 1: Introduce $q_\phi(z\vert{}x)$ and Take the Expectation

Since $\log p_\theta(x)$ does not depend on $z$, its expectation with respect to any probability distribution $q_\phi(z\vert{}x)$ over $z$ is simply itself:

$$\log p_\theta(x) = \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x) \right]$$

Using the definition of conditional probability, we can rewrite the joint distribution $p_\theta(x, z)$ as:

$$p_\theta(x) = \frac{p_\theta(x, z)}{p_\theta(z\vert{}x)}$$

Substitute this expression into the expectation:

$$\log p_\theta(x) = \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x, z)}{p_\theta(z\vert{}x)} \right) \right]$$

---

## Step 2: Expand with $q_\phi(z\vert{}x)$ Inside the Fraction

Multiply the numerator and denominator inside the logarithm by $q_\phi(z\vert{}x)$:

$$\log p_\theta(x) = \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x, z)}{p_\theta(z\vert{}x)} \cdot \frac{q_\phi(z\vert{}x)}{q_\phi(z\vert{}x)} \right) \right]$$

Rearrange the terms inside the logarithm:

$$\log p_\theta(x) = \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x, z)}{q_\phi(z\vert{}x)} \cdot \frac{q_\phi(z\vert{}x)}{p_\theta(z\vert{}x)} \right) \right]$$

---

## Step 3: Split the Logarithm using Log Rules

Recall the logarithm product rule $\log(A \cdot B) = \log A + \log B$:

$$\log p_\theta(x) = \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x, z)}{q_\phi(z\vert{}x)} \right) + \log \left( \frac{q_\phi(z\vert{}x)}{p_\theta(z\vert{}x)} \right) \right]$$

By the linearity of expectation ($\mathbb{E}[A + B] = \mathbb{E}[A] + \mathbb{E}[B]$):

$$\log p_\theta(x) = \underbrace{\mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x, z)}{q_\phi(z\vert{}x)} \right) \right]}_{\text{Term 1: ELBO}} + \underbrace{\mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{q_\phi(z\vert{}x)}{p_\theta(z\vert{}x)} \right) \right]}_{\text{Term 2: KL Divergence}}$$

---

## Step 4: Identify the Terms

### Analyzing Term 2:

By definition, the Kullback-Leibler (KL) divergence between two continuous distributions $q(z)$ and $p(z)$ is:

$$D_{\text{KL}}(q \parallel p) = \int q(z) \log \left( \frac{q(z)}{p(z)} \right) dz = \mathbb{E}_{q} \left[ \log \left( \frac{q(z)}{p(z)} \right) \right]$$

Therefore, Term 2 is precisely the KL divergence between our approximate posterior $q_\phi(z\vert{}x)$ and the true posterior $p_\theta(z\vert{}x)$:

$$\mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{q_\phi(z\vert{}x)}{p_\theta(z\vert{}x)} \right) \right] = D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p_\theta(z\vert{}x) \right)$$

### Analyzing Term 1 (ELBO):

Term 1 is defined as the **Evidence Lower Bound (ELBO)**, denoted as $\mathcal{L}_{\text{ELBO}}(\theta, \phi; x)$:

$$\mathcal{L}_{\text{ELBO}}(\theta, \phi; x) = \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x, z)}{q_\phi(z\vert{}x)} \right) \right]$$

Putting both terms back together gives the fundamental identity:

$$\log p_\theta(x) = \mathcal{L}_{\text{ELBO}}(\theta, \phi; x) + D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p_\theta(z\vert{}x) \right)$$

---

## Step 5: Why is it a "Lower Bound"?

A key property of KL divergence is Gibbs' Inequality, which states that KL divergence is **always non-negative**:

$$D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p_\theta(z\vert{}x) \right) \ge 0$$

Because the KL term is $\ge 0$, dropping it yields an inequality:

$$\log p_\theta(x) \ge \mathcal{L}_{\text{ELBO}}(\theta, \phi; x)$$

This proves that $\mathcal{L}_{\text{ELBO}}$ is a strict **lower bound** on the true marginal log-likelihood (the "evidence") $\log p_\theta(x)$.

---

## Step 6: Deconstructing ELBO into Reconstruction + Regularization

To make the ELBO practical to optimize with neural networks, we further expand Term 1.

Using $p_\theta(x, z) = p_\theta(x\vert{}z) p(z)$:

$$\begin{aligned} \mathcal{L}_{\text{ELBO}}(\theta, \phi; x) &= \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p_\theta(x\vert{}z) p(z)}{q_\phi(z\vert{}x)} \right) \right] \\ &= \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x\vert{}z) + \log \left( \frac{p(z)}{q_\phi(z\vert{}x)} \right) \right] \\ &= \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x\vert{}z) \right] + \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{p(z)}{q_\phi(z\vert{}x)} \right) \right] \end{aligned}$$

Using the rule $\log\left(\frac{A}{B}\right) = -\log\left(\frac{B}{A}\right)$ on the second log term:

$$\begin{aligned} \mathcal{L}_{\text{ELBO}}(\theta, \phi; x) &= \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x\vert{}z) \right] - \mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log \left( \frac{q_\phi(z\vert{}x)}{p(z)} \right) \right] \\ &= \underbrace{\mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x\vert{}z) \right]}_{\text{Reconstruction Term}} - \underbrace{D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p(z) \right)}_{\text{Prior Regularization Term}} \end{aligned}$$

---

## Summary of the Final Formulation

| Term | Mathematical Expression | Intuition |
| --- | --- | --- |
| **Reconstruction Loss** | $\mathbb{E}_{q_\phi(z\vert{}x)} [\log p_\theta(x\Vert{}z)]$ | Measures how accurately the decoder $p_\theta(x\Vert{}z)$ reconstructs $x$ from latent code $z$. |
| **KL Regularization** | $D_{\text{KL}}(q_\phi(z\Vert{}x) \parallel p(z))$ | Measures how much the encoded distribution $q_\phi(z\Vert{}x)$ deviates from the prior $p(z) = \mathcal{N}(0, I)$. |

Since neural networks minimize loss via gradient descent, we negate the ELBO to get our final training objective:

$$\text{Loss}_{\text{VAE}}(\theta, \phi; x) = -\mathbb{E}_{q_\phi(z\vert{}x)} \left[ \log p_\theta(x\vert{}z) \right] + D_{\text{KL}}\left( q_\phi(z\vert{}x) \parallel p(z) \right)$$

| Autoencoder Variant | Key Mechanism | Best Used For |
| --- | --- | --- |
| **Undercomplete AE** | Latent dimension is smaller than input dimension. | Basic dimensionality reduction & compression. |
| **Denoising AE (DAE)** | Adds noise to inputs during training; network learns to strip noise away. | Noise removal & robust feature extraction. |
| **Sparse AE** | Forces hidden units to be mostly inactive via regularization penalties. | Feature extraction without shrinking layer size. |
| **Variational AE (VAE)** | Encodes inputs into probability distributions (mean and variance) rather than fixed points. | Generative tasks (e.g., generating new realistic images). |
| **Convolutional AE (CAE)** | Replaces fully connected layers with convolutional and upsampling layers. | Processing image and spatial data. |

---

## 4. Key Applications

* **Dimensionality Reduction:** Serves as a non-linear alternative to Principal Component Analysis (PCA).
* **Anomaly & Outlier Detection:** If trained on normal data, the autoencoder will produce high reconstruction errors on unusual inputs.
* **Image Denoising:** Restores clean signals from corrupted inputs.
* **Pre-training & Representation Learning:** Learns rich representations from unlabeled data to initialize supervised models.

<p style="page-break-after:always;"></p>